# ICS 3202: Artificial Intelligence  Practical Exercise 4
## Developing a Knowledge-Based Self-Driving Trailer Agent

**Submission notebook:** `Autonomous_Trailer.ipynb`  
**Repository:** `KnowledgeEngineering`  
**Group members:** *Insert names and student numbers before submission*  
**Course:** ICS 3202 (Aug–Nov 2026)

### Objective
Build an **explainable knowledge-based agent (KBA)** that models a self-driving cargo trailer transporting cargo from Nairobi to Kisumu. It uses the directed graph and route distances supplied in **Practical Exercise 3**, stores explicit facts in a knowledge base (KB), applies **IF–THEN inference rules**, and chooses actions based on road conditions and trailer safety.

> **Scope and accuracy:** This is a transparent educational **planning and advisory simulation**, **not** vehicle-control software. Distances are carried over from the group's Exercise 3 artefacts (not freshly checked against Google Maps). The original network is a *single directed chain*, so its 352 km path is optimal only **within that supplied graph**, not proven globally optimal. Weather, traffic, sensor values and hazards used below are **demonstration scenarios**, not live observations.


## 1. Reuse of Practical Exercise 3: PEAS and task environment

| PEAS element | Self-driving trailer design |
|---|---|
| **Performance** | Collision avoidance, legal compliance, cargo safety, fuel efficiency and timely delivery |
| **Environment** | Kenyan highways, towns, road users, weighbridges, weather and junctions |
| **Actuators (conceptual)** | Steering, braking, throttle, lights, indicators, driver/control-centre alerts |
| **Sensors (conceptual)** | GPS/GNSS, cameras, radar, LiDAR, IMU, hitch-angle, tyre, load, weather/traffic feed |

**Task environment:** partially observable, stochastic, sequential, dynamic and multiagent. We therefore maintain knowledge about the current world state and reassess actions when new percepts arrive.

**Architecture:** `percepts / supplied road facts → knowledge base → forward-chaining rule engine → safety decision → UCS route planner → explanation / advisory action`.


In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Dict, List, Set, Tuple, Optional
import heapq
import math

print("Knowledge-based trailer demonstration ready (standard Python library only).")

Knowledge-based trailer demonstration ready (standard Python library only).


## 2. Road facts: directed graph and distance knowledge

The following **14 directed edges and their costs** are transcribed from the group's Exercise 3 notebook and design document. Costs are in kilometres. Edges have no implied reverse direction. Cumulative distance is calculated from the source route; do not mistake those values for verified real-time distances.


In [2]:
ROAD_GRAPH: Dict[str, Dict[str, float]] = {
    "Nairobi": {"Rungiri": 19.0},
    "Rungiri": {"Gitaru": 4.1},
    "Gitaru": {"Limuru": 15.0},
    "Limuru": {"Kimende Township": 14.5},
    "Kimende Township": {"Gilgil": 68.4},
    "Gilgil": {"Nakuru East": 30.0},
    "Nakuru East": {"Salga": 37.0},
    "Salga": {"Kericho": 78.0},
    "Kericho": {"Kapsoit": 11.0},
    "Kapsoit": {"Kaitui": 11.0},
    "Kaitui": {"Awasi": 20.0},
    "Awasi": {"Ahero": 18.0},
    "Ahero": {"Orongo": 15.0},
    "Orongo": {"Kisumu": 11.0},
    "Kisumu": {},
}

ROUTE = list(ROAD_GRAPH)
assert len(ROAD_GRAPH) == 15
assert sum(len(out) for out in ROAD_GRAPH.values()) == 14

cumulative = 0.0
for city in ROUTE:
    print(f"{city:<19} {cumulative:>6.1f} km from Nairobi")
    cumulative += sum(ROAD_GRAPH[city].values())
assert math.isclose(cumulative, 352.0, abs_tol=1e-9)
print(f"\nExercise 3 route total: {cumulative:.1f} km")

Nairobi                0.0 km from Nairobi
Rungiri               19.0 km from Nairobi
Gitaru                23.1 km from Nairobi
Limuru                38.1 km from Nairobi
Kimende Township      52.6 km from Nairobi
Gilgil               121.0 km from Nairobi
Nakuru East          151.0 km from Nairobi
Salga                188.0 km from Nairobi
Kericho              266.0 km from Nairobi
Kapsoit              277.0 km from Nairobi
Kaitui               288.0 km from Nairobi
Awasi                308.0 km from Nairobi
Ahero                326.0 km from Nairobi
Orongo               341.0 km from Nairobi
Kisumu               352.0 km from Nairobi

Exercise 3 route total: 352.0 km


## 3. Knowledge representation: facts and rule base

A knowledge base records both **static facts** (roads and distances) and **dynamic percepts** (weather, obstacles, trailer condition). Facts are represented as Python data structures with explicit meanings. The inference engine adds **derived facts** and records which rule produced each conclusion.

| Example fact | Meaning |
|---|---|
| `weather = "heavy_rain"` | Rain reported by a simulated weather percept |
| `obstacle = True` | Obstacle detected on intended segment |
| `hitch_angle_deg = 42` | Trailer articulation angle measured by a simulated sensor |
| `road_blocked = True` | Traversal of an edge is prohibited for this scenario |
| `decision = "STOP_AND_ALERT"` | Rule-based recommendation |

**Priority:** emergency safety rules override route planning and comfort. Thresholds below are **illustrative classroom choices**, not engineering safety certifications or regulations.


In [3]:
@dataclass
class Percepts:
    location: str = "Nairobi"
    weather: str = "clear"       # clear, light_rain, heavy_rain, fog
    traffic: str = "normal"      # normal, heavy
    obstacle: bool = False
    hitch_angle_deg: float = 0.0
    tyre_pressure_ok: bool = True
    cargo_secured: bool = True
    fuel_percent: float = 70.0
    road_blocked: bool = False

@dataclass
class KnowledgeBase:
    roads: Dict[str, Dict[str, float]]
    percepts: Percepts = field(default_factory=Percepts)
    facts: Set[str] = field(default_factory=set)
    explanations: List[str] = field(default_factory=list)

    def observe(self, new_percepts: Percepts) -> None:
        if new_percepts.location not in self.roads:
            raise ValueError(f"Unknown town: {new_percepts.location}")
        if new_percepts.weather not in {"clear", "light_rain", "heavy_rain", "fog"}:
            raise ValueError("Unrecognised weather value")
        if new_percepts.traffic not in {"normal", "heavy"}:
            raise ValueError("Unrecognised traffic value")
        if not (0 <= new_percepts.fuel_percent <= 100):
            raise ValueError("Fuel percentage must be 0..100")
        if not (0 <= new_percepts.hitch_angle_deg <= 180):
            raise ValueError("Hitch angle must be 0..180 degrees")
        self.percepts = new_percepts
        self.facts = set()
        self.explanations = []

    def assert_fact(self, name: str, reason: str) -> None:
        if name not in self.facts:
            self.facts.add(name)
            self.explanations.append(f"{name}: {reason}")

    def infer(self) -> Set[str]:
        p = self.percepts
        # Forward chaining: percepts -> derived condition facts -> decision facts.
        if p.weather in {"heavy_rain", "fog"}:
            self.assert_fact("REDUCED_VISIBILITY_OR_GRIP", "Rule R1: severe weather observed")
        if p.traffic == "heavy":
            self.assert_fact("CONGESTION", "Rule R2: traffic is heavy")
        if p.obstacle:
            self.assert_fact("OBSTACLE_AHEAD", "Rule R3: obstacle reported")
        if p.road_blocked:
            self.assert_fact("ROAD_BLOCKED", "Rule R4: current outgoing road reported blocked")
        if p.hitch_angle_deg >= 40:
            self.assert_fact("JACKKNIFE_RISK", "Rule R5: simulated hitch angle >= 40 degrees")
        if not p.tyre_pressure_ok:
            self.assert_fact("TYRE_FAULT", "Rule R6: tyre-pressure check failed")
        if not p.cargo_secured:
            self.assert_fact("UNSECURED_CARGO", "Rule R7: cargo restraint check failed")
        if p.fuel_percent < 15:
            self.assert_fact("LOW_FUEL", "Rule R8: fuel below illustrative 15% threshold")
        if self.facts & {"JACKKNIFE_RISK", "TYRE_FAULT", "UNSECURED_CARGO", "OBSTACLE_AHEAD"}:
            self.assert_fact("STOP_AND_ALERT", "Rule R9: immediate safety issue overrides travel")
        elif "ROAD_BLOCKED" in self.facts:
            self.assert_fact("REPLAN_OR_WAIT", "Rule R10: intended road is blocked")
        elif "LOW_FUEL" in self.facts:
            self.assert_fact("REFUEL_ADVISORY", "Rule R11: low fuel needs service planning")
        elif "REDUCED_VISIBILITY_OR_GRIP" in self.facts or "CONGESTION" in self.facts:
            self.assert_fact("PROCEED_WITH_CAUTION", "Rule R12: adverse travel conditions")
        else:
            self.assert_fact("PROCEED_NORMALLY", "Rule R13: no modeled hazard present")
        return self.facts

    def decision(self) -> str:
        for item in ("STOP_AND_ALERT", "REPLAN_OR_WAIT", "REFUEL_ADVISORY", "PROCEED_WITH_CAUTION", "PROCEED_NORMALLY"):
            if item in self.facts:
                return item
        raise RuntimeError("Call infer() before decision()")

## 4. Inference rules and conflict resolution

The rule engine uses **forward chaining**: each new percept is matched to `IF condition THEN fact` rules, and the resulting facts trigger a decision rule. Conflicts are resolved by explicit priority: `STOP_AND_ALERT` → `REPLAN_OR_WAIT` → `REFUEL_ADVISORY` → `PROCEED_WITH_CAUTION` → `PROCEED_NORMALLY`. The explanations act as a proof trace showing **why** the agent reached its conclusion.

A blocked road is evaluated separately from an immediate physical hazard; the agent does **not** falsely claim a detour when the supplied graph has none.


## 5. Route planning using Uniform Cost Search (UCS)

UCS removes the lowest-cost frontier state first. With non-negative edge distances, it returns the least-distance path in the **modeled network**. The optional `blocked_edges` set removes impassable road links. For the current one-path graph, a blockage means no reachable route; genuine rerouting would require independently sourced alternative edges.


In [4]:
def uniform_cost_search(graph: Dict[str, Dict[str, float]], start: str, goal: str,
                        blocked_edges: Optional[Set[Tuple[str, str]]] = None):
    if start not in graph or goal not in graph:
        raise ValueError("Start and goal must appear in the knowledge base")
    blocked_edges = blocked_edges or set()
    frontier = [(0.0, start, [start])]
    best_cost = {start: 0.0}
    expanded = []
    while frontier:
        cost, town, path = heapq.heappop(frontier)
        if cost > best_cost.get(town, float("inf")):
            continue
        expanded.append((town, round(cost, 1)))
        if town == goal:
            return {"path": path, "distance_km": round(cost, 1), "expanded": expanded}
        for neighbor, distance in graph[town].items():
            if (town, neighbor) in blocked_edges:
                continue
            if distance < 0:
                raise ValueError("UCS requires nonnegative distances")
            candidate = cost + distance
            if candidate < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = candidate
                heapq.heappush(frontier, (candidate, neighbor, path + [neighbor]))
    return {"path": None, "distance_km": None, "expanded": expanded}

normal_route = uniform_cost_search(ROAD_GRAPH, "Nairobi", "Kisumu")
print("Planned route:", " -> ".join(normal_route["path"]))
print("Total distance:", normal_route["distance_km"], "km")
print("UCS expansions:", len(normal_route["expanded"]))

Planned route: Nairobi -> Rungiri -> Gitaru -> Limuru -> Kimende Township -> Gilgil -> Nakuru East -> Salga -> Kericho -> Kapsoit -> Kaitui -> Awasi -> Ahero -> Orongo -> Kisumu
Total distance: 352.0 km
UCS expansions: 15


## 6. Integrating planning, sensing, reasoning, and action selection

An agent cycle consists of `OBSERVE → UPDATE KB → INFER → PLAN → ADVISE/EXPLAIN`. Its output represents **advice** that an actual vehicle would need to validate against real safety-certified control systems. This demonstration does not send physical steering or brake commands.


In [5]:
def trailer_agent(percepts: Percepts, destination: str = "Kisumu") -> dict:
    kb = KnowledgeBase(roads=ROAD_GRAPH)
    kb.observe(percepts)
    kb.infer()
    action = kb.decision()
    blocked = set()
    if percepts.road_blocked:
        blocked = {(percepts.location, nxt) for nxt in ROAD_GRAPH[percepts.location]}
    route = uniform_cost_search(ROAD_GRAPH, percepts.location, destination, blocked)

    if action == "STOP_AND_ALERT":
        message = "Safety alert: stop in a safe manner and notify the operator; do not continue automatically."
    elif action == "REPLAN_OR_WAIT":
        message = ("Road unavailable: no alternate route in the supplied graph; await updated map or operator guidance."
                   if route["path"] is None else "Blocked road: use the available alternative route after verification.")
    elif action == "REFUEL_ADVISORY":
        message = "Plan a verified fuel stop before continuing; no station information is stored."
    elif action == "PROCEED_WITH_CAUTION":
        message = "Travel cautiously; adjust speed and separation to verified road conditions."
    else:
        message = "No hazard detected by the model; route can be considered for travel."
    return {"location": percepts.location,
            "destination": destination,
            "action": action,
            "advice": message,
            "route": route["path"],
            "remaining_distance_km": route["distance_km"],
            "derived_facts": sorted(kb.facts),
            "explanation": kb.explanations}

def show_decision(result: dict) -> None:
    print(f"Location: {result['location']}  |  Goal: {result['destination']}")
    print(f"Decision: {result['action']}")
    print(f"Advice: {result['advice']}")
    if result["route"]:
        print("Route:", " -> ".join(result["route"]))
        print("Remaining route distance:", result["remaining_distance_km"], "km")
    else:
        print("Route: unavailable in supplied directed graph")
    print("Explanation (rule trace):")
    for reason in result["explanation"]:
        print(" -", reason)

show_decision(trailer_agent(Percepts()))

Location: Nairobi  |  Goal: Kisumu
Decision: PROCEED_NORMALLY
Advice: No hazard detected by the model; route can be considered for travel.
Route: Nairobi -> Rungiri -> Gitaru -> Limuru -> Kimende Township -> Gilgil -> Nakuru East -> Salga -> Kericho -> Kapsoit -> Kaitui -> Awasi -> Ahero -> Orongo -> Kisumu
Remaining route distance: 352.0 km
Explanation (rule trace):
 - PROCEED_NORMALLY: Rule R13: no modeled hazard present


## 7. Demonstration: knowledge-based responses to changing conditions

The following inputs are **synthetic** to demonstrate that the same route-planning agent changes decisions when its knowledge changes. They do not represent observations of actual Kenyan road conditions.


In [6]:
SCENARIOS = {
    "A — clear road": Percepts(),
    "B — heavy rain": Percepts(weather="heavy_rain", location="Gilgil"),
    "C — obstacle ahead": Percepts(obstacle=True, location="Kericho"),
    "D — trailer articulation danger": Percepts(hitch_angle_deg=45, location="Salga"),
    "E — low fuel": Percepts(fuel_percent=9, location="Nakuru East"),
    "F — road closure": Percepts(road_blocked=True, location="Ahero"),
    "G — several hazards": Percepts(weather="fog", tyre_pressure_ok=False, fuel_percent=8),
}
for label, scenario in SCENARIOS.items():
    result = trailer_agent(scenario)
    print(f"\n{label}: {result['action']}; remaining distance = {result['remaining_distance_km']}")
    print(" ", result["advice"])


A — clear road: PROCEED_NORMALLY; remaining distance = 352.0
  No hazard detected by the model; route can be considered for travel.

B — heavy rain: PROCEED_WITH_CAUTION; remaining distance = 231.0
  Travel cautiously; adjust speed and separation to verified road conditions.

C — obstacle ahead: STOP_AND_ALERT; remaining distance = 86.0
  Safety alert: stop in a safe manner and notify the operator; do not continue automatically.

D — trailer articulation danger: STOP_AND_ALERT; remaining distance = 164.0
  Safety alert: stop in a safe manner and notify the operator; do not continue automatically.

E — low fuel: REFUEL_ADVISORY; remaining distance = 201.0
  Plan a verified fuel stop before continuing; no station information is stored.

F — road closure: REPLAN_OR_WAIT; remaining distance = None
  Road unavailable: no alternate route in the supplied graph; await updated map or operator guidance.

G — several hazards: STOP_AND_ALERT; remaining distance = 352.0
  Safety alert: stop in a s

## 8. Optional user interaction (text-based, Jupyter-compatible)

To try a custom scenario, change the `Percepts(...)` parameters in the next cell and run it. This avoids requiring online APIs, API keys or interactive prompts during grading. A real system could feed `observe()` with permitted GPS, traffic or weather APIs, with data provenance and timestamps; **no API or scraper was used in this notebook**.


In [7]:
# EDIT these example fields to simulate your own sensor readings.
user_scenario = Percepts(
    location="Nairobi",
    weather="light_rain",
    traffic="heavy",
    obstacle=False,
    hitch_angle_deg=5,
    tyre_pressure_ok=True,
    cargo_secured=True,
    fuel_percent=60,
    road_blocked=False,
)
show_decision(trailer_agent(user_scenario))

Location: Nairobi  |  Goal: Kisumu
Decision: PROCEED_WITH_CAUTION
Advice: Travel cautiously; adjust speed and separation to verified road conditions.
Route: Nairobi -> Rungiri -> Gitaru -> Limuru -> Kimende Township -> Gilgil -> Nakuru East -> Salga -> Kericho -> Kapsoit -> Kaitui -> Awasi -> Ahero -> Orongo -> Kisumu
Remaining route distance: 352.0 km
Explanation (rule trace):
 - CONGESTION: Rule R2: traffic is heavy
 - PROCEED_WITH_CAUTION: Rule R12: adverse travel conditions


## 9. Validation and tests

These assertions check the Exercise 3 distance, validity of the route, correct application of inference rules, handling of unsafe conditions, and correct reporting when a road closure disconnects the supplied graph.


In [8]:
def test_agent():
    route = uniform_cost_search(ROAD_GRAPH, "Nairobi", "Kisumu")
    assert route["path"] == ROUTE
    assert math.isclose(route["distance_km"], 352.0)
    assert trailer_agent(Percepts())["action"] == "PROCEED_NORMALLY"
    assert trailer_agent(Percepts(weather="fog"))["action"] == "PROCEED_WITH_CAUTION"
    assert trailer_agent(Percepts(traffic="heavy"))["action"] == "PROCEED_WITH_CAUTION"
    assert trailer_agent(Percepts(obstacle=True))["action"] == "STOP_AND_ALERT"
    assert trailer_agent(Percepts(tyre_pressure_ok=False))["action"] == "STOP_AND_ALERT"
    assert trailer_agent(Percepts(cargo_secured=False))["action"] == "STOP_AND_ALERT"
    assert trailer_agent(Percepts(hitch_angle_deg=42))["action"] == "STOP_AND_ALERT"
    assert trailer_agent(Percepts(fuel_percent=5))["action"] == "REFUEL_ADVISORY"
    assert trailer_agent(Percepts(road_blocked=True))["action"] == "REPLAN_OR_WAIT"
    assert trailer_agent(Percepts(road_blocked=True))["route"] is None
    assert trailer_agent(Percepts(location="Kisumu"))["remaining_distance_km"] == 0
    # Safety takes precedence over low fuel, rain and traffic.
    assert trailer_agent(Percepts(tyre_pressure_ok=False, fuel_percent=4, weather="fog"))["action"] == "STOP_AND_ALERT"
    try:
        trailer_agent(Percepts(location="Unknown"))
        raise AssertionError("Unknown location should fail")
    except ValueError:
        pass
    print("PASS: 13 validation checks completed.")

test_agent()

PASS: 13 validation checks completed.


## 10. Discussion, limitations and conclusion

**Knowledge engineering process:** (1) acquire the route facts from Exercise 3, (2) represent them as directed weighted edges and sensor-state facts, (3) encode safety rules, (4) infer new conditions from percepts, (5) make a prioritized decision, (6) compute a path with UCS, and (7) produce an explanation trace.

**What this demonstrates:** explicit knowledge representation; a working **knowledge base**; **forward-chaining inference**; rule-based decision-making with conflict resolution; graph search and path costs; a simple **sense–think–act** cycle; and tests for multiple changing scenarios.

**Limitations:** The network has only one directed route, so genuine detouring cannot yet be demonstrated. The distance entries are inherited coursework data, not independently revalidated. The rules, weather/traffic inputs and hitch-angle threshold are educational abstractions. No perception pipeline, realtime API ingestion, certified braking controller, legal speed-limit database or road-trailer engineering validation is included.

**Future improvements:** add verified alternative roads and directionality; attach verified date-stamped traffic/weather feeds where permitted; model weighbridge and trailer clearance restrictions; implement uncertainty handling and route-cost updates; and add physical control only in an approved simulator with appropriate safety review.

### Sources and attribution
1. **Group Exercise 3**, `UCS_Trailer.ipynb` — exact road graph, UCS algorithm pattern and 352 km path cost (user-supplied course material).
2. **Group Exercise 3**, `design(1).md` — PEAS, environment classification, town sequence, cumulative-distance derivation and path cost (user-supplied course material).
3. **ICS 3202, Practical Exercise 4 brief** — exercise requirements and required notebook/repository naming (user-supplied instructions).




## Groupwork Evidence

The following photograph shows the group members who
collaborated on Practical Exercise 4.

![Groupwork Evidence](<AI GROUP WORK.jpeg>)